# 实验：vLLM + Agent 天气工具调用

> 这份 notebook 用一个完全本地的模拟天气工具，测试 `vLLM` 后端和 `agent/` 侧的 OpenAI-compatible tool calling 是否打通。

> 不依赖检索语料，也不依赖外部天气 API。


## 0. 环境与服务说明

假设你已经在服务器上启动好了 vLLM，并为当前模型配置了可用的 tool parser。

这个 notebook 的目标很简单：

1. 给模型一个天气问题
2. 让模型调用 `get_weather`
3. 本地执行模拟工具
4. 把工具结果回填给模型，观察最终回答


In [1]:
!python --version
!pip install -r agent/requirements.txt


Python 3.10.17
Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://mirrors.huaweicloud.com/repository/pypi/simple

[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


## 1. 初始化 vLLM 客户端

这里的 `MODEL_NAME` 可以切换成你当前起好的模型名，比如：

- `pangu_auto`
- `qwen_auto`


In [2]:
from pathlib import Path
import json
import sys

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from agent.vllm_client import VLLMClient

VLLM_BASE_URL = 'http://127.0.0.1:8000/v1'
# MODEL_NAME = 'pangu_auto'
MODEL_NAME = 'qwen_auto'
API_KEY = 'dummy'

client = VLLMClient(base_url=VLLM_BASE_URL, api_key=API_KEY)


## 2. 定义一个完全本地的模拟天气工具

为了让测试稳定，我们把天气数据写死在 notebook 里。


In [3]:
MOCK_WEATHER_DB = {
    '南京': {'temperature_c': 24, 'condition': '多云', 'humidity': 58, 'wind': '东北风 3级'},
    '上海': {'temperature_c': 27, 'condition': '晴', 'humidity': 67, 'wind': '东南风 4级'},
    '北京': {'temperature_c': 21, 'condition': '晴转阴', 'humidity': 42, 'wind': '西北风 2级'},
    '杭州': {'temperature_c': 25, 'condition': '小雨', 'humidity': 81, 'wind': '东风 2级'},
}


def get_weather(city: str):
    city = city.strip()
    if city not in MOCK_WEATHER_DB:
        return {
            'city': city,
            'found': False,
            'message': 'mock weather data not found',
        }
    return {
        'city': city,
        'found': True,
        **MOCK_WEATHER_DB[city],
    }


tool_specs = [
    {
        'type': 'function',
        'function': {
            'name': 'get_weather',
            'description': 'Get mock weather data for a given city.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'city': {
                        'type': 'string',
                        'description': 'City name in Chinese, such as 南京 or 上海.',
                    }
                },
                'required': ['city'],
            },
        },
    }
]

tool_registry = {
    'get_weather': get_weather,
}

print(json.dumps(tool_specs, ensure_ascii=False, indent=2))


[
  {
    "type": "function",
    "function": {
      "name": "get_weather",
      "description": "Get mock weather data for a given city.",
      "parameters": {
        "type": "object",
        "properties": {
          "city": {
            "type": "string",
            "description": "City name in Chinese, such as 南京 or 上海."
          }
        },
        "required": [
          "city"
        ]
      }
    }
  }
]


## 3. 工具执行辅助函数


In [4]:
def execute_tool_call(tool_call, registry):
    function = tool_call.get('function', {})
    name = function.get('name', '')
    arguments = function.get('arguments', '{}')
    if isinstance(arguments, str):
        arguments = json.loads(arguments)
    result = registry[name](**arguments)
    return {
        'tool_name': name,
        'arguments': arguments,
        'tool_result': result,
    }


## 4. 最小 agent loop

这里故意把系统提示写得很明确：只要用户问天气，必须先调用 `get_weather`，不能靠常识直接回答。


In [5]:
def run_weather_agent(query: str, max_rounds: int = 4, max_tokens: int = 512):
    messages = [
        {
            'role': 'system',
            'content': (
                'You are a weather assistant. '
                'For any weather-related question, you must call the get_weather tool before answering. '
                'Do not answer weather questions from memory. '
                'After receiving tool results, answer in Chinese and keep the answer concise.'
            ),
        },
        {'role': 'user', 'content': query},
    ]
    trajectory = []

    for round_id in range(1, max_rounds + 1):
        response = client.simple_chat(
            model=MODEL_NAME,
            messages=messages,
            temperature=0.0,
            max_tokens=max_tokens,
            tools=tool_specs,
            tool_choice='auto',
        )
        message = response['choices'][0]['message']
        raw_content = message.get('content', '')
        tool_calls = message.get('tool_calls') or []

        step = {
            'round_id': round_id,
            'assistant_content': raw_content,
            'tool_calls': tool_calls,
            'usage': response.get('usage', {}),
        }
        trajectory.append(step)

        assistant_message = {'role': 'assistant', 'content': raw_content}
        if tool_calls:
            assistant_message['tool_calls'] = tool_calls
        messages.append(assistant_message)

        if not tool_calls:
            return {
                'query': query,
                'status': 'completed',
                'final_output': raw_content,
                'trajectory': trajectory,
                'messages': messages,
            }

        tool_results = []
        for tool_call in tool_calls:
            executed = execute_tool_call(tool_call, tool_registry)
            tool_results.append(executed)
            messages.append(
                {
                    'role': 'tool',
                    'tool_call_id': tool_call['id'],
                    'content': json.dumps(executed['tool_result'], ensure_ascii=False),
                }
            )
        step['tool_results'] = tool_results

    return {
        'query': query,
        'status': 'max_rounds_reached',
        'final_output': '',
        'trajectory': trajectory,
        'messages': messages,
    }


## 5. 单条测试

这个例子通常会触发两次工具调用：先查南京，再查上海。


In [6]:
demo_query = '请比较南京和上海今天的天气，告诉我哪个更适合穿短袖，并简要说明原因。'
demo = run_weather_agent(demo_query, max_rounds=4)

print('status:', demo['status'])
print('\nfinal output:\n')
print(demo['final_output'])
print('\ntrajectory preview:\n')
print(json.dumps(demo['trajectory'], ensure_ascii=False, indent=2)[:6000])


status: completed

final output:

<think>
好的，现在我需要比较南京和上海的天气，判断哪个更适合穿短袖。首先看温度，南京是24度，上海是27度。上海温度更高，但南京的天气是多云，上海是晴天。湿度方面，南京58%，上海67%，上海湿度稍高但差异不大。风力的话，南京东北风3级，上海东南风4级。总体来看，上海温度更高且天气晴朗，可能更适合穿短袖。不过需要考虑湿度是否会影响体感温度，但用户只要求简要说明，所以主要依据温度和天气状况。结论应该是上海更适合穿短袖。
</think>

上海今天的气温为27℃，天气晴朗，体感较为舒适；南京气温24℃，多云天气。由于上海温度更高且无降水，更适合穿短袖。

trajectory preview:

[
  {
    "round_id": 1,
    "assistant_content": "<think>\n好的，用户让我比较南京和上海今天的天气，看看哪个更适合穿短袖，还需要简要说明原因。首先，我需要调用get_weather工具来获取这两个城市的天气数据。不过用户提到的是今天的天气，所以参数应该是城市名称，但工具的参数里只接受一个城市名。可能需要分别调用两次，一次南京，一次上海。\n\n然后，根据返回的数据，比如温度、天气状况（比如是否下雨或高温），来判断哪个城市的天气更适合穿短袖。通常来说，温度较高且天气晴朗的地方更适合短袖。如果其中一个城市温度明显高于另一个，或者有高温预警，那可能更合适。另外，如果上海今天有雨，而南京晴天，可能南京更适合穿短袖。不过具体还要看实际数据。\n\n所以，首先需要获取南京和上海的天气信息，然后比较温度和天气情况，再给出建议。现在需要调用两次get_weather函数，分别传入南京和上海作为参数。\n</think>\n\n",
    "tool_calls": [
      {
        "id": "chatcmpl-tool-5606d35ee9614d22ada2af80dc8862f9",
        "type": "function",
        "function": {
          "name": "get_weather",
          "arguments": "{\"city\": \"南京\"}

## 6. 再试一个单城市问题


In [7]:
single_city_demo = run_weather_agent('南京今天什么天气？', max_rounds=4)
print(single_city_demo['final_output'])


<think>
好的，用户问南京今天的天气，我需要先调用get_weather工具来获取最新数据。根据之前的工具调用，返回的数据显示南京今天多云，温度24摄氏度，湿度58%，东北风3级。现在需要把这些信息用中文简洁地组织起来回答用户。确保包含温度、天气状况、湿度和风力信息，保持回答自然流畅。用户可能关心是否带伞或者穿衣建议，但问题只问了天气情况，所以保持回答紧扣提供的数据即可。检查是否有遗漏的参数，比如是否需要提醒温度单位，但工具返回的温度已经是摄氏度，所以不需要额外说明。最后确认回答符合要求，不使用Markdown，保持口语化。
</think>

南京今天多云，气温24摄氏度，湿度58%，东北风3级。
